# Lonfall test
This is similar to the landfall_DT.ipynb script but instead of landfall we detect AR masks across the 126 W longitude.

In [2]:
import xarray as xa
import itertools
import numpy as np
import matplotlib.pyplot as plt
from dask.distributed import Client
from dask_jobqueue import PBSCluster
import time
import pandas as pd
import gc

In [3]:
cluster = PBSCluster(
    account='WYOM0169',
    queue='main',
    cores=4,
    memory='64GB',
    walltime='08:00:00',
    job_extra_directives=['-o /glade/u/home/tcorrie/dask_worker_log/', '-e /glade/u/home/tcorrie/dask_worker_log/'],
    )
cluster.scale(64)
client = Client(cluster)

In [5]:
print(client)

<Client: 'tcp://128.117.208.192:37585' processes=64 threads=64, memory=0.93 TiB>


In [6]:
def squeeze_dataset(ds):
    ds_cond = ds.squeeze(drop=True).rename({'shapemap', 'ARmask'})
    return ds_cond

In [7]:
def process_timestep(tstep, ardt, model, member):
    ARmask = xa.open_dataset(f'/glade/work/tcorrie/ARdata/ARmasks/{ardt}/ARmask.{model}.{member}.nc', use_cftime=True)
    #ARmask = squeeze_dataset(ARmask) if ardt == 'tARget' else ARmask
    #land = xa.open_dataset('/glade/derecho/scratch/tcorrie/regrids/landmask_regridded.nc')
    wall = ARmask.sel(lon=-126, method='nearest').sel(lat=slice(32.5,48.51))['ARmask']
    wall_tstep = wall.isel(time=tstep)

    ARmaskwall = wall_tstep.values.astype('int').tolist()

    # Check for minimum WUS landfall (1 deg, 10 pixels) in range of 32.5 to 48.5
    lonfall_widths = [(key, len(list(group))) for key, group in itertools.groupby(ARmaskwall) if key == 1]
    lonfall_extent = [lw[1]/10 for lw in lonfall_widths]

    if max(lonfall_extent, default=0) < 1.0:
        return (0, 0, 0, 0, 0, ARmaskwall) # Order: WUS, SCA, NCA, OR, WA

    def check_region(lat_min, lat_max):
        cl = wall_tstep.sel(lat=slice(lat_min, lat_max))
        amc = cl.values.astype('int').tolist()
        widths = [(key, len(list(group))) for key, group in itertools.groupby(amc) if key == 1]
        extents = [lw[1]/10 for lw in widths]
        return 1 if max(extents, default=0) >= 0.5 else 0
        
    lonfall_sca = check_region(32.5, 37.21) 
    lonfall_nca = check_region(37.2,42.01)
    lonfall_or = check_region(42.0,46.31)
    lonfall_wa = check_region(46.3,48.51)
    return(1, lonfall_sca, lonfall_nca, lonfall_or, lonfall_wa, ARmaskwall)

In [8]:
wusd3 = pd.read_csv('../wusd3_gcms.csv', index_col=0)
for i in range(1011, 1201, 20):
    wusd3.loc[len(wusd3)] = ['cesm2-le', 'CESM2-LE', str(i), '365_day']
wusd3

,Model,Model Name,Member,Calendar
0,access-cm2,ACCESS-CM2,r5i1p1f1,standard
1,canesm5,CanESM5,r1i1p2f1,365_day
2,cesm2,CESM2,r11i1p1f1,365_day
3,cnrm-esm2-1,CNRM-ESM2-1,r1i1p1f2,standard
4,ec-earth3,EC-Earth3,r1i1p1f1,standard
5,ec-earth3-veg,EC-Earth3-Veg,r1i1p1f1,standard
6,era5,ERA5,NaN,standard
7,fgoals-g3,FGOALS-g3,r1i1p1f1,365_day
8,giss-e2-1-g,GISS-E2-1-G,r1i1p1f2,365_day
9,miroc6,MIROC6,r1i1p1f1,standard


In [9]:
for row in range(len(wusd3)):
    if row <= 19:
        continue

    model = wusd3.loc[row, 'Model']
    member = wusd3.loc[row, 'Member']
    print(f"Lonfall for {model}/{member};")
    
    for ardt in ['G18', 'TE', 'tARget']: #G18, TE, tARget
        print(f'{ardt}:', end=" ")
        t1 = time.time()
        ARmask = xa.open_dataset(f'/glade/work/tcorrie/ARdata/ARmasks/{ardt}/ARmask.{model}.{member}.nc', use_cftime=True)
        ARmask = ARmask.chunk({'time':'auto', 'lat':-1, 'lon':-1})
        wall = ARmask.sel(lon=-126, method='nearest').sel(lat=slice(32.5,48.51))
        wall_lats = wall.lat.data
        times = ARmask.time.values
    
        #ARmask_futures = client.scatter(ARmask, broadcast=True)
        
        futures = client.map(process_timestep, range(len(times)), ardt=ardt, model=model, member=member)    
        results = client.gather(futures)
        
        lonfalls_wus, lonfalls_sca, lonfalls_nca, lonfalls_or, lonfalls_wa, ARmaskwalls = zip(*results)
        
        lonfall_ds = xa.Dataset(
                coords=dict(time=('time',times)),
                data_vars=dict(
                    lonfall_wus=(["time"], np.array(lonfalls_wus)),
                    lonfall_sca=(["time"], np.array(lonfalls_sca)),
                    lonfall_nca=(["time"], np.array(lonfalls_nca)),
                    lonfall_or=(["time"], np.array(lonfalls_or)),
                    lonfall_wa=(["time"], np.array(lonfalls_wa)),
                ),
                attrs=dict(description='A binary mask for discerning lonfalling ARs over the Western US and split by region')
                
            )
    
        wall_mask_ds = xa.Dataset(
            coords=dict(
                time=('time', times),
                lat=('lat', wall_lats)
            ),
            data_vars=dict(
                ARwall=(['time','lat'], np.array(ARmaskwalls))
            ),
            attrs=dict(description='The points where an AR mask overlaps a specific longitude.')
        )
        
        lonfall_ds.to_netcdf(f'/glade/work/tcorrie/ARdata/ARlonfall/{ardt}/ARlonfall.{ardt}.{model}.{member}.nc')
        wall_mask_ds.to_netcdf(f'/glade/work/tcorrie/ARdata/ARwall/{ardt}/ARwall.{ardt}.{model}.{member}.nc')
        t2 = time.time()
        print(f'{(t2-t1)//60:.0f}\'{(t2-t1)%60:04.1f}\" run time')
        gc.collect()

Lonfall for cesm2-le/1091;
G18: 3'24.8" run time
TE: 4'18.2" run time
tARget: 7'59.5" run time
Lonfall for cesm2-le/1111;
G18: 3'57.2" run time
TE: 4'24.4" run time
tARget: 8'09.2" run time
Lonfall for cesm2-le/1131;
G18: 3'54.3" run time
TE: 4'30.6" run time
tARget: 8'15.3" run time
Lonfall for cesm2-le/1151;
G18: 4'09.1" run time
TE: 4'38.3" run time
tARget: 8'19.9" run time
Lonfall for cesm2-le/1171;
G18: 4'09.2" run time
TE: 4'40.1" run time
tARget: 8'24.8" run time
Lonfall for cesm2-le/1191;
G18: 4'10.3" run time
TE: 4'39.3" run time
tARget: 8'26.1" run time


In [ ]:
print(cluster)

In [ ]:
print(futures[0].status)

In [ ]:
lonfall_ds.isel(time=365*46+134)

In [ ]:
ARmask

In [ ]:
wall.lat.data

In [ ]:
process_timestep(665, ardt='G18', model='access-cm2', member='r5i1p1f1')

In [ ]:
future = client.submit(process_timestep, 0, ardt='G18', model='access-cm2', member='r5i1p1f1')
client.gather(future)

In [ ]:
def debug_timestep(tstep, ardt, model, member):
    ARmask = xa.open_dataset(f'/glade/work/tcorrie/ARdata/ARmasks/{ardt}/ARmask.{model}.{member}.nc', use_cftime=True)
    wall = ARmask.sel(lon=-126, method='nearest').sel(lat=slice(32.5, 48.51))
    print(type(wall))
    print(wall)
    print(wall.data_vars)
    return (type(wall), wall, wall.data_vars)

future = client.submit(debug_timestep, 0, ardt='G18', model='access-cm2', member='r5i1p1f1')
client.gather(future)